# MarsLandformNet V3.1 — DINOv2-LoRA (Anti-Overfit Edition)

**V1 Problem**: Train F1=0.98 vs Val F1=0.57 → massive overfitting

**V2 Fixes**:
1. **Backbone fully frozen** — only LoRA adapters trainable (0.9M vs 14M params)
2. **MixUp augmentation** (α=0.3) — interpolate samples to smooth decision boundaries
3. **Stronger dropout** (0.5) + **higher weight decay** (0.05)
4. **WeightedRandomSampler** — oversample CCF/LVF minority classes
5. **Simpler head** — 2 layers instead of 3 to reduce capacity
6. **Stronger augmentation** — RandomErasing, GaussianNoise
7. **Label smoothing 0.1** — prevent overconfident predictions
8. **EMA** (Exponential Moving Average) — stabilize generalization

## 0. Setup

In [ ]:
!pip install -q transformers peft accelerate timm pillow scikit-learn matplotlib seaborn

In [ ]:
import os
import json
import copy
import time
import math
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from PIL import Image
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f}GB)')
else:
    print('WARNING: No GPU! Runtime > Change runtime type > T4 GPU')
print(f'PyTorch: {torch.__version__}, Device: {device}')

## 1. Download Data

In [ ]:
DATA_DIR = Path('/content/v3_data')
DATA_DIR.mkdir(exist_ok=True)

GITHUB_TOKEN = ''  # Set if repo is private

RELEASE_TAG = 'v3-training-e2e'
ASSET_NAME = 'v3_colab_e2e_data.tar.gz'
TAR_PATH = DATA_DIR / ASSET_NAME

if not (DATA_DIR / 'tile_labels_v3.json').exists():
    print('Downloading training data (~740MB)...')
    if GITHUB_TOKEN:
        import requests
        headers = {'Authorization': f'token {GITHUB_TOKEN}', 'Accept': 'application/vnd.github+json'}
        r = requests.get(f'https://api.github.com/repos/jejuchild/MarsLab/releases/tags/{RELEASE_TAG}', headers=headers)
        assets = r.json().get('assets', [])
        asset = next((a for a in assets if a['name'] == ASSET_NAME), None)
        assert asset, f'Asset {ASSET_NAME} not found in release {RELEASE_TAG}'
        asset_url = asset['url']
        !wget -q --show-progress --header='Authorization: token {GITHUB_TOKEN}' --header='Accept: application/octet-stream' -O {TAR_PATH} {asset_url}
    else:
        url = f'https://github.com/jejuchild/MarsLab/releases/download/{RELEASE_TAG}/{ASSET_NAME}'
        !wget -q --show-progress -O {TAR_PATH} {url}
    
    assert TAR_PATH.exists() and TAR_PATH.stat().st_size > 1_000_000, 'Download failed!'
    print(f'Downloaded: {TAR_PATH.stat().st_size/1e6:.0f}MB')
    print('Extracting...')
    !tar xzf {TAR_PATH} -C {DATA_DIR}
    !rm -f {TAR_PATH}
    print('Done!')
else:
    print('Data already extracted.')

print(f'\nContents of {DATA_DIR}:')
for f in sorted(DATA_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name}: {f.stat().st_size/1e6:.1f}MB')
    elif f.is_dir():
        n = sum(1 for _ in f.rglob('*.jpg'))
        print(f'  {f.name}/: {n} JPEGs')

## 2. Configuration

In [ ]:
CFG = {
    # Model
    'model_name': 'facebook/dinov2-base',
    'hidden_dim': 768,
    'mola_dim': 25,
    'num_classes': 4,
    'head_hidden': 128,       # Smaller head (was 256)
    'dropout': 0.5,           # Heavier dropout (was 0.3)
    
    # LoRA
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.15,     # More LoRA dropout (was 0.1)
    'lora_targets': ['query', 'key', 'value'],
    'unfreeze_last_n_blocks': 0,  # FROZEN backbone (was 2)
    
    # Training
    'batch_size': 64,
    'num_epochs': 80,
    'lr_backbone': 3e-4,      # Higher LR for LoRA-only (was 5e-5)
    'lr_head': 1e-3,
    'weight_decay': 0.05,     # Stronger decay (was 0.01)
    'warmup_epochs': 5,       # Longer warmup (was 3)
    'label_smoothing': 0.1,   # More smoothing (was 0.05)
    
    # MixUp
    'mixup_alpha': 0.3,
    
    # EMA
    'ema_decay': 0.999,
    
    # Data
    'tile_size': 224,
    'num_workers': 2,
    'class_names': ['LDA', 'LVF', 'CCF', 'OTHER'],
    'class_to_idx': {'LDA': 0, 'LVF': 1, 'CCF': 2, 'OTHER': 3},
    
    # Augmentation
    'aug_hflip': True,
    'aug_vflip': True,
    'aug_rotation': True,
    'aug_color_jitter': 0.3,   # Stronger (was 0.2)
    'aug_random_erasing': 0.25,
    'aug_gaussian_noise': 0.02,
    
    # Paths
    'data_dir': '/content/v3_data',
    'save_dir': '/content/checkpoints',
}

print('Key changes from V1:')
print('  unfreeze_last_n_blocks: 2 -> 0 (backbone frozen)')
print('  dropout: 0.3 -> 0.5')
print('  weight_decay: 0.01 -> 0.05')
print('  label_smoothing: 0.05 -> 0.1')
print('  head_hidden: 256 -> 128')
print('  + MixUp (alpha=0.3)')
print('  + EMA (decay=0.999)')
print('  + WeightedRandomSampler')
print('  + RandomErasing + GaussianNoise')

## 3. Dataset with Strong Augmentation

In [ ]:
class MarsLandformDataset(Dataset):
    def __init__(self, tile_labels, split_indices, mola_features, tile_index,
                 data_dir, class_to_idx, transform=None):
        self.data_dir = Path(data_dir)
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.mola_features = mola_features
        self.tile_index = tile_index
        
        self.samples = []
        for idx in split_indices:
            t = tile_labels[idx]
            if t['label'] == 'UNLABELED':
                continue
            self.samples.append(t)
        
        print(f'  Dataset: {len(self.samples)} samples')
        labels = [s['label'] for s in self.samples]
        for cls_name in sorted(class_to_idx.keys()):
            n = sum(1 for l in labels if l == cls_name)
            print(f'    {cls_name}: {n} ({100*n/len(labels):.1f}%)')
    
    def __len__(self):
        return len(self.samples)
    
    def _load_tile_image(self, sample):
        img_id = sample['image_id']
        tr, tc = sample['tile_row'], sample['tile_col']
        tile_key = f'{img_id}_{tr}_{tc}'
        rel_path = self.tile_index.get(tile_key)
        if rel_path:
            img_path = self.data_dir / rel_path
        else:
            img_path = self.data_dir / 'tiles' / img_id / f'tile_{tr:03d}_{tc:03d}.jpg'
        try:
            img = Image.open(img_path).convert('RGB')
            img = np.array(img, dtype=np.float32) / 255.0
        except Exception:
            img = np.zeros((224, 224, 3), dtype=np.float32)
        return img
    
    def _get_mola(self, sample):
        img_id = sample['image_id']
        tile_key = f"{sample['tile_row']}_{sample['tile_col']}"
        img_mola = self.mola_features.get(img_id, {})
        if tile_key in img_mola:
            return img_mola[tile_key].astype(np.float32)
        return np.zeros(25, dtype=np.float32)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = self._load_tile_image(sample)
        if self.transform:
            img = self.transform(img)
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img = (img - mean) / std
        img_tensor = torch.from_numpy(img.transpose(2, 0, 1))
        mola = torch.from_numpy(self._get_mola(sample))
        label = self.class_to_idx[sample['label']]
        return img_tensor, mola, label


class TrainAugmentation:
    def __init__(self, cfg):
        self.hflip = cfg['aug_hflip']
        self.vflip = cfg['aug_vflip']
        self.rotation = cfg['aug_rotation']
        self.jitter = cfg['aug_color_jitter']
        self.erase_prob = cfg.get('aug_random_erasing', 0)
        self.noise_std = cfg.get('aug_gaussian_noise', 0)
    
    def __call__(self, img):
        # Random 90-degree rotation
        if self.rotation:
            k = random.randint(0, 3)
            if k > 0:
                img = np.rot90(img, k=k, axes=(0, 1)).copy()
        
        # Random flips
        if self.hflip and random.random() > 0.5:
            img = np.fliplr(img).copy()
        if self.vflip and random.random() > 0.5:
            img = np.flipud(img).copy()
        
        # Color jitter (brightness + contrast)
        if self.jitter > 0:
            brightness = 1.0 + random.uniform(-self.jitter, self.jitter)
            img = img * brightness
            contrast = 1.0 + random.uniform(-self.jitter, self.jitter)
            mean_val = img.mean()
            img = (img - mean_val) * contrast + mean_val
            img = np.clip(img, 0.0, 1.0)
        
        # Gaussian noise
        if self.noise_std > 0:
            noise = np.random.normal(0, self.noise_std, img.shape).astype(np.float32)
            img = np.clip(img + noise, 0.0, 1.0)
        
        # Random erasing
        if self.erase_prob > 0 and random.random() < self.erase_prob:
            h, w = img.shape[:2]
            eh = random.randint(h // 8, h // 3)
            ew = random.randint(w // 8, w // 3)
            y = random.randint(0, h - eh)
            x = random.randint(0, w - ew)
            img[y:y+eh, x:x+ew] = np.random.uniform(0, 1, (eh, ew, 3)).astype(np.float32)
        
        return img

In [ ]:
# Load data
data_dir = Path(CFG['data_dir'])

print('Loading data...')
with open(data_dir / 'tile_labels_v3.json') as f:
    tile_labels = json.load(f)
with open(data_dir / 'tile_splits_v3.json') as f:
    splits = json.load(f)
with open(data_dir / 'tile_index.json') as f:
    tile_index = json.load(f)
mola_features = np.load(data_dir / 'mola_features_by_tile.npy', allow_pickle=True).item()

print(f'Total tiles: {len(tile_labels)}')
print(f'Splits - Train: {len(splits["train"])}, Val: {len(splits["val"])}, Test: {len(splits["test"])}')

train_aug = TrainAugmentation(CFG)

print('\nTrain:')
train_ds = MarsLandformDataset(
    tile_labels, splits['train'], mola_features, tile_index,
    data_dir, CFG['class_to_idx'], transform=train_aug
)
print('Val:')
val_ds = MarsLandformDataset(
    tile_labels, splits['val'], mola_features, tile_index,
    data_dir, CFG['class_to_idx'], transform=None
)
print('Test:')
test_ds = MarsLandformDataset(
    tile_labels, splits['test'], mola_features, tile_index,
    data_dir, CFG['class_to_idx'], transform=None
)

In [ ]:
# WeightedRandomSampler — oversample minority classes
train_labels = [CFG['class_to_idx'][s['label']] for s in train_ds.samples]
label_counts = Counter(train_labels)
n_samples = len(train_labels)

# Weight per sample = 1 / class_count
class_sample_weights = {cls: n_samples / count for cls, count in label_counts.items()}
sample_weights = [class_sample_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=n_samples, replacement=True)

print('Sample weights per class:')
for cls_idx, name in enumerate(CFG['class_names']):
    w = class_sample_weights.get(cls_idx, 0)
    n = label_counts.get(cls_idx, 0)
    print(f'  {name}: weight={w:.2f} (n={n})')

# Dataloaders
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], sampler=sampler,
                          num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                        num_workers=CFG['num_workers'], pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False,
                         num_workers=CFG['num_workers'], pin_memory=True)

print(f'\nBatches/epoch - Train: {len(train_loader)}, Val: {len(val_loader)}')

## 4. Model (Simpler Head + Full Freeze)

In [ ]:
from transformers import Dinov2Model
from peft import LoraConfig, get_peft_model


class MarsLandformNetV3(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        
        # DINOv2 backbone
        print('Loading DINOv2-base...')
        self.backbone = Dinov2Model.from_pretrained(cfg['model_name'])
        
        # Apply LoRA
        print('Applying LoRA adapters...')
        lora_config = LoraConfig(
            r=cfg['lora_r'],
            lora_alpha=cfg['lora_alpha'],
            lora_dropout=cfg['lora_dropout'],
            target_modules=cfg['lora_targets'],
            bias='none',
        )
        self.backbone = get_peft_model(self.backbone, lora_config)
        self.backbone.print_trainable_parameters()
        
        # NO unfreezing — only LoRA adapters are trainable in backbone
        
        # MOLA projection
        self.mola_bn = nn.BatchNorm1d(cfg['mola_dim'])
        self.mola_proj = nn.Linear(cfg['mola_dim'], 64)
        
        # Simpler classification head (2 layers instead of 3)
        combined_dim = cfg['hidden_dim'] + 64  # 832
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, cfg['head_hidden']),
            nn.BatchNorm1d(cfg['head_hidden']),
            nn.GELU(),
            nn.Dropout(cfg['dropout']),
            nn.Linear(cfg['head_hidden'], cfg['num_classes']),
        )
    
    def forward(self, pixel_values, mola_features):
        outputs = self.backbone(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0]  # (B, 768)
        
        mola = self.mola_bn(mola_features)
        mola = F.gelu(self.mola_proj(mola))  # (B, 64)
        
        combined = torch.cat([cls_token, mola], dim=1)  # (B, 832)
        logits = self.classifier(combined)
        return logits

In [ ]:
def load_ssl_lora_weights(model, ssl_path):
    ckpt = torch.load(ssl_path, map_location='cpu')
    ssl_state = ckpt['lora_state_dict']
    mapped = {}
    for ssl_key, tensor in ssl_state.items():
        peft_key = ssl_key.replace('backbone.', '', 1)
        mapped[peft_key] = tensor
    result = model.backbone.load_state_dict(mapped, strict=False)
    loaded = len(mapped) - len(result.unexpected_keys)
    print(f'SSL LoRA weights: {loaded}/{len(mapped)} tensors matched')
    return model


class EMA:
    """Exponential Moving Average of model parameters."""
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    
    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.shadow[name] = self.decay * self.shadow[name] + (1 - self.decay) * param.data
    
    def apply_shadow(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]
    
    def restore(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data = self.backup[name]
        self.backup = {}

In [ ]:
# Build model
model = MarsLandformNetV3(CFG)

ssl_path = data_dir / 'ssl_lora_weights.pt'
if ssl_path.exists():
    model = load_ssl_lora_weights(model, ssl_path)
else:
    print('No SSL weights — training from scratch')

model = model.to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal: {total/1e6:.1f}M, Trainable: {trainable/1e6:.2f}M ({100*trainable/total:.1f}%)')

# EMA
ema = EMA(model, decay=CFG['ema_decay'])

## 5. Training Setup

In [ ]:
# Class weights for loss (sqrt-inverse, NO sampler weights — sampler handles balance)
total_train = len(train_ds)
label_dist = Counter([s['label'] for s in train_ds.samples])
class_weights = []
for name in CFG['class_names']:
    count = label_dist.get(name, 1)
    w = math.sqrt(total_train / count)
    class_weights.append(w)
mean_w = sum(class_weights) / len(class_weights)
class_weights = [w / mean_w for w in class_weights]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

# NOTE: V1 had both sampler AND loss weights — that's double-boosting.
# Here we use sampler for balanced batches, loss weights are mild (sqrt).
# The sampler ensures ~25% each class per batch,
# loss weights give slight extra push to rarer classes.
print('Loss weights:')
for name, w in zip(CFG['class_names'], class_weights):
    print(f'  {name}: {w:.3f}')

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=CFG['label_smoothing']
)

# Optimizer
backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and 'backbone' in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and 'backbone' not in n]

print(f'\nBackbone trainable: {sum(p.numel() for p in backbone_params)/1e6:.2f}M')
print(f'Head trainable: {sum(p.numel() for p in head_params)/1e6:.2f}M')

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': CFG['lr_backbone']},
    {'params': head_params, 'lr': CFG['lr_head']},
], weight_decay=CFG['weight_decay'])

# Cosine with warmup
total_steps = CFG['num_epochs'] * len(train_loader)
warmup_steps = CFG['warmup_epochs'] * len(train_loader)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))  # min LR = 1% of peak

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler()

print(f'\nTotal steps: {total_steps}, Warmup: {warmup_steps}')

## 6. MixUp + Training Loop

In [ ]:
def mixup_data(x, mola, y, alpha=0.3):
    """MixUp: interpolate between random pairs."""
    if alpha <= 0:
        return x, mola, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)  # Ensure lambda >= 0.5
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    mixed_mola = lam * mola + (1 - lam) * mola[index]
    return mixed_x, mixed_mola, y, y[index], lam


def mixup_criterion(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, ema, device, mixup_alpha):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for images, mola, labels in loader:
        images = images.to(device, non_blocking=True)
        mola = mola.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        # MixUp
        mixed_images, mixed_mola, targets_a, targets_b, lam = mixup_data(
            images, mola, labels, alpha=mixup_alpha
        )
        
        optimizer.zero_grad()
        
        with autocast():
            logits = model(mixed_images, mixed_mola)
            loss = mixup_criterion(criterion, logits, targets_a, targets_b, lam)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        # EMA update
        ema.update(model)
        
        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(targets_a.cpu().numpy())  # Use targets_a for reporting
    
    avg_loss = total_loss / len(all_labels)
    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    return avg_loss, f1, acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for images, mola, labels in loader:
        images = images.to(device, non_blocking=True)
        mola = mola.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        with autocast():
            logits = model(images, mola)
            loss = criterion(logits, labels)
        
        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(all_labels)
    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    return avg_loss, f1, acc, all_preds, all_labels

In [ ]:
# Mount drive (optional)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    save_dir = Path('/content/drive/MyDrive/marslandform_v3')
    save_dir.mkdir(parents=True, exist_ok=True)
    CFG['save_dir'] = str(save_dir)
    print(f'Saving to Drive: {save_dir}')
except Exception:
    save_dir = Path(CFG['save_dir'])
    save_dir.mkdir(parents=True, exist_ok=True)
    print(f'Saving locally: {save_dir}')
    print('Download from file browser when done.')

In [ ]:
# ─── TRAINING ──────────────────────────────────────────────────────────────
best_val_f1 = 0.0
patience = 15
patience_counter = 0
history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': [],
           'val_f1_ema': [], 'lr': []}

save_dir = Path(CFG['save_dir'])

print(f'\n{"="*75}')
print(f'Training MarsLandformNet V3.1 (Anti-Overfit) for {CFG["num_epochs"]} epochs')
print(f'{"="*75}\n')

for epoch in range(1, CFG['num_epochs'] + 1):
    t0 = time.time()
    
    # Train
    train_loss, train_f1, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, scaler, ema,
        device, CFG['mixup_alpha']
    )
    
    # Validate (normal weights)
    val_loss, val_f1, val_acc, val_preds, val_labels = evaluate(
        model, val_loader, criterion, device
    )
    
    # Validate with EMA weights
    ema.apply_shadow(model)
    _, val_f1_ema, _, val_preds_ema, val_labels_ema = evaluate(
        model, val_loader, criterion, device
    )
    ema.restore(model)
    
    # Use best of EMA vs normal
    use_ema = val_f1_ema > val_f1
    best_this_epoch = max(val_f1, val_f1_ema)
    
    elapsed = time.time() - t0
    current_lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    history['val_f1_ema'].append(val_f1_ema)
    history['lr'].append(current_lr)
    
    improved = ''
    if best_this_epoch > best_val_f1:
        best_val_f1 = best_this_epoch
        patience_counter = 0
        improved = ' ★ BEST'
        
        # Save best with the better weights
        if use_ema:
            ema.apply_shadow(model)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_f1': best_this_epoch,
            'val_acc': val_acc,
            'cfg': CFG,
            'used_ema': use_ema,
        }, save_dir / 'best_model.pt')
        if use_ema:
            ema.restore(model)
    else:
        patience_counter += 1
    
    ema_tag = f' EMA={val_f1_ema:.4f}' if abs(val_f1_ema - val_f1) > 0.001 else ''
    print(f'Ep {epoch:3d}/{CFG["num_epochs"]} | '
          f'Train L={train_loss:.4f} F1={train_f1:.4f} | '
          f'Val L={val_loss:.4f} F1={val_f1:.4f}{ema_tag} | '
          f'LR={current_lr:.2e} | {elapsed:.0f}s{improved}')
    
    if epoch % 10 == 0:
        # Use EMA or normal for reporting
        report_preds = val_preds_ema if use_ema else val_preds
        report_labels = val_labels_ema if use_ema else val_labels
        per_class = f1_score(report_labels, report_preds, average=None)
        for i, name in enumerate(CFG['class_names']):
            print(f'    {name}: F1={per_class[i]:.4f}')
    
    if epoch % 20 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
            'cfg': CFG,
        }, save_dir / f'checkpoint_epoch{epoch}.pt')
    
    if patience_counter >= patience:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs)')
        break

print(f'\n{"="*75}')
print(f'Training complete! Best val F1: {best_val_f1:.4f}')
print(f'{"="*75}')

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_f1'], label='Train', linewidth=2)
axes[1].plot(history['val_f1'], label='Val', linewidth=2)
axes[1].plot(history['val_f1_ema'], label='Val (EMA)', linewidth=2, linestyle='--')
axes[1].axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Macro F1')
axes[1].set_title('F1 Score'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(history['lr'], linewidth=2, color='green')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].set_title('Learning Rate'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(save_dir / 'training_curves_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Test Evaluation

In [ ]:
# Load best model
best_ckpt = torch.load(save_dir / 'best_model.pt', map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])
print(f'Best model: epoch {best_ckpt["epoch"]}, val F1={best_ckpt["val_f1"]:.4f}, EMA={best_ckpt.get("used_ema", False)}')

test_loss, test_f1, test_acc, test_preds, test_labels = evaluate(
    model, test_loader, criterion, device
)

print(f'\n{"="*50}')
print(f'TEST SET RESULTS')
print(f'{"="*50}')
print(f'Macro F1: {test_f1:.4f}')
print(f'Accuracy: {test_acc:.4f}')
print(f'{"="*50}\n')
print(classification_report(test_labels, test_preds,
                            target_names=CFG['class_names'], digits=4))

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
cm_pct = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CFG['class_names'], yticklabels=CFG['class_names'], ax=ax1)
ax1.set_xlabel('Predicted'); ax1.set_ylabel('True')
ax1.set_title('Confusion Matrix (Counts)')

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=CFG['class_names'], yticklabels=CFG['class_names'], ax=ax2)
ax2.set_xlabel('Predicted'); ax2.set_ylabel('True')
ax2.set_title('Confusion Matrix (%)')

plt.tight_layout()
plt.savefig(save_dir / 'confusion_matrix_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Export for MarsLab

In [ ]:
deploy_state = {
    'model_state_dict': model.state_dict(),
    'cfg': CFG,
    'class_names': CFG['class_names'],
    'class_to_idx': CFG['class_to_idx'],
    'test_f1': test_f1,
    'test_acc': test_acc,
    'epoch': best_ckpt['epoch'],
}
deploy_path = save_dir / 'marslandform_v3_deploy.pt'
torch.save(deploy_state, deploy_path)
print(f'Deploy checkpoint: {deploy_path} ({deploy_path.stat().st_size/1e6:.1f}MB)')
print(f'Test F1={test_f1:.4f}, Acc={test_acc:.4f}')
print(f'\nDownload from file browser or Google Drive: {CFG["save_dir"]}')

## 10. Error Analysis

In [ ]:
misclassified = [(i, p, t) for i, (p, t) in enumerate(zip(test_preds, test_labels)) if p != t]
print(f'Misclassified: {len(misclassified)}/{len(test_preds)} ({100*len(misclassified)/len(test_preds):.1f}%)')

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
random.shuffle(misclassified)
for i, ax in enumerate(axes.flat):
    if i >= len(misclassified):
        ax.axis('off'); continue
    idx, pred, true = misclassified[i]
    img_t, _, _ = test_ds[idx]
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_display = (img_t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img_display)
    ax.set_title(f'T:{CFG["class_names"][true]} P:{CFG["class_names"][pred]}', fontsize=9, color='red')
    ax.axis('off')
plt.suptitle('Misclassified Tiles', fontsize=14)
plt.tight_layout()
plt.savefig(save_dir / 'errors_v2.png', dpi=150, bbox_inches='tight')
plt.show()